# AOA Feature Engineering

This notebook turns the refined AoA dataset into training-ready feature sets.



## 1. Goal

We build two staged feature sets:
- `FeatureSet-A`: core numeric and morphology-aware features
- `FeatureSet-B`: `FeatureSet-A` plus cleaned categorical features



In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Find the project root and set input/output paths.
# The output folder is used to save the final feature files.
ROOT = Path.cwd()
if ROOT.name == 'iteration2':
    ROOT = ROOT.parent.parent

INPUT_PATH = ROOT / 'data/processed/AoA_refined_dataset_v3_freq2.xlsx'
OUTPUT_DIR = ROOT / 'data/processed/iteration2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH


PosixPath('/Users/datong/Documents/5120/Nurodiversity inclusive design/data/TP10_DS/data/processed/AoA_refined_dataset_v3_freq2.xlsx')

## 2. Load data

We use `v3_freq2` because it keeps a good balance between row count and class distribution.



In [2]:
# Load the selected AoA dataset for feature engineering.
df = pd.read_excel(INPUT_PATH).copy()

print('raw shape:', df.shape)
display(df.head())


raw shape: (9997, 20)


,Word,Alternative.spelling,Freq_pm,Dom_PoS_SUBTLEX,Nletters,Nphon,Nsyll,Lemma_highest_PoS,AoA_Kup,Perc_known,AoA_Kup_lem,Perc_known_lem,AoA_Bird_lem,AoA_Bristol_lem,AoA_Cort_lem,AoA_Schock,Perc_known_final,AoA_final,Age_Group,AoA_source
0,a,a,20415.274510,Article,1,1,1,a,2.893384,1.000000,2.893384,1.000000,3.156247,NaN,NaN,NaN,1.000000,2.893384,below_6,AoA_Kup
1,abandon,abandon,8.098039,Verb,7,7,3,abandon,8.320000,1.000000,8.320000,1.000000,NaN,NaN,NaN,NaN,1.000000,8.320000,age_8_plus,AoA_Kup
2,abandoned,abandoned,13.294118,Verb,9,8,3,abandon,NaN,NaN,8.320000,1.000000,NaN,NaN,NaN,NaN,1.000000,8.320000,age_8_plus,AoA_Kup_lem
3,abbey,abbey,3.176471,Noun,5,3,2,abbey,13.060000,0.857143,13.060000,0.857143,NaN,9.48740,NaN,11.292186,0.857143,13.060000,age_8_plus,AoA_Kup
4,abdomen,abdomen,3.352941,Noun,7,7,3,abdomen,8.610000,1.000000,8.610000,1.000000,NaN,10.92228,NaN,NaN,1.000000,8.610000,age_8_plus,AoA_Kup


## 3. Basic cleanup

We clean noisy categorical values before creating features. In particular, `Dom_PoS_SUBTLEX` contains a small number of numeric entries that should be treated as invalid.



In [3]:
# Repair noisy POS values and standardise some text columns.
# This makes later categorical features more stable.
numeric_pos_mask = df['Dom_PoS_SUBTLEX'].apply(lambda x: isinstance(x, (int, float)) and not pd.isna(x))
df.loc[numeric_pos_mask, 'Dom_PoS_SUBTLEX'] = 'Unknown'

df['Dom_PoS_SUBTLEX'] = df['Dom_PoS_SUBTLEX'].fillna('Unknown').astype(str)
df['AoA_source'] = df['AoA_source'].fillna('Unknown').astype(str)
df['Word'] = df['Word'].astype(str)
df['Lemma_highest_PoS'] = df['Lemma_highest_PoS'].astype(str)

print('rows with repaired POS values:', int(numeric_pos_mask.sum()))
print('POS unique values:', df['Dom_PoS_SUBTLEX'].nunique())


rows with repaired POS values: 23
POS unique values: 18


## 4. Feature engineering

The new features are grouped into four categories:
- frequency transformation
- ratio and interaction features
- lemma and morphology features
- cleaned categorical features


In [4]:
# Create new features from frequency, word structure, and lemma information.
# These are the main engineered features used for later modelling.
# Frequency feature
df['log_freq_pm'] = np.log1p(df['Freq_pm'])

# Ratio and interaction features
df['letters_per_syll'] = df['Nletters'] / df['Nsyll'].replace(0, np.nan)
df['phon_per_syll'] = df['Nphon'] / df['Nsyll'].replace(0, np.nan)
df['letters_per_phon'] = df['Nletters'] / df['Nphon'].replace(0, np.nan)
df['letters_x_syll'] = df['Nletters'] * df['Nsyll']
df['phon_x_syll'] = df['Nphon'] * df['Nsyll']

# Lemma and morphology features
df['is_lemma_match'] = (
    df['Word'].str.lower() == df['Lemma_highest_PoS'].str.lower()
).astype(int)
df['lemma_length'] = df['Lemma_highest_PoS'].str.len()
df['lemma_length_diff'] = df['Word'].str.len() - df['lemma_length']

# Optional helper features
df['pos_group'] = df['Dom_PoS_SUBTLEX'].replace({
    'Article': 'Function',
    'Conjunction': 'Function',
    'Determiner': 'Function',
    'Preposition': 'Function',
    'Pronoun': 'Function',
    'Interjection': 'Function',
    'Noun': 'Content',
    'Verb': 'Content',
    'Adjective': 'Content',
    'Adverb': 'Content',
    'Name': 'Content',
    'Number': 'Other',
    'Unknown': 'Other',
}).fillna('Other')

engineered_cols = [
    'log_freq_pm',
    'letters_per_syll',
    'phon_per_syll',
    'letters_per_phon',
    'letters_x_syll',
    'phon_x_syll',
    'is_lemma_match',
    'lemma_length_diff',
    'pos_group',
]

display(df[engineered_cols].head())


,log_freq_pm,letters_per_syll,phon_per_syll,letters_per_phon,letters_x_syll,phon_x_syll,is_lemma_match,lemma_length_diff,pos_group
0,9.924088,1.000000,1.000000,1.000000,1,1,1,0,Function
1,2.208059,2.333333,2.333333,1.000000,21,21,1,0,Content
2,2.659848,3.000000,2.666667,1.125000,27,24,0,2,Content
3,1.429467,2.500000,1.500000,1.666667,10,6,1,0,Content
4,1.470852,2.333333,2.333333,1.000000,21,21,1,0,Content


## 5. Define feature sets

`FeatureSet-A` is the first recommended training set.
`FeatureSet-B` adds cleaned categorical features for a second experiment.



In [5]:
# Build two staged feature sets.
# FeatureSet-A is the main numeric set, and FeatureSet-B adds cleaned categories.
feature_set_a = [
    'log_freq_pm',
    'Nletters',
    'Nphon',
    'Nsyll',
    'Perc_known_final',
    'letters_per_syll',
    'phon_per_syll',
    'letters_per_phon',
    'letters_x_syll',
    'phon_x_syll',
    'is_lemma_match',
    'lemma_length_diff',
]

feature_set_b = feature_set_a + [
    'Dom_PoS_SUBTLEX',
    'pos_group',
    'AoA_source',
]

target_cols = ['AoA_final', 'Age_Group']
id_cols = ['Word']

feature_set_a_df = df[id_cols + feature_set_a + target_cols].copy()
feature_set_b_df = df[id_cols + feature_set_b + target_cols].copy()

print('FeatureSet-A shape:', feature_set_a_df.shape)
print('FeatureSet-B shape:', feature_set_b_df.shape)
display(feature_set_a_df.head())


FeatureSet-A shape: (9997, 15)
FeatureSet-B shape: (9997, 18)


,Word,log_freq_pm,Nletters,Nphon,Nsyll,Perc_known_final,letters_per_syll,phon_per_syll,letters_per_phon,letters_x_syll,phon_x_syll,is_lemma_match,lemma_length_diff,AoA_final,Age_Group
0,a,9.924088,1,1,1,1.000000,1.000000,1.000000,1.000000,1,1,1,0,2.893384,below_6
1,abandon,2.208059,7,7,3,1.000000,2.333333,2.333333,1.000000,21,21,1,0,8.320000,age_8_plus
2,abandoned,2.659848,9,8,3,1.000000,3.000000,2.666667,1.125000,27,24,0,2,8.320000,age_8_plus
3,abbey,1.429467,5,3,2,0.857143,2.500000,1.500000,1.666667,10,6,1,0,13.060000,age_8_plus
4,abdomen,1.470852,7,7,3,1.000000,2.333333,2.333333,1.000000,21,21,1,0,8.610000,age_8_plus


## 6. Quick quality checks

Before exporting, we check missing values and class balance.



In [6]:
# Check missing values and label balance before export.
missing_a = feature_set_a_df.isna().mean().sort_values(ascending=False).to_frame('missing_ratio')
missing_b = feature_set_b_df.isna().mean().sort_values(ascending=False).to_frame('missing_ratio')

print('FeatureSet-A missing ratio')
display(missing_a.head(15))

print('FeatureSet-B missing ratio')
display(missing_b.head(15))

print('Age group distribution')
display((df['Age_Group'].value_counts(normalize=True) * 100).round(2).to_frame('percentage'))


FeatureSet-A missing ratio


,missing_ratio
Word,0.0
log_freq_pm,0.0
Nletters,0.0
Nphon,0.0
Nsyll,0.0
Perc_known_final,0.0
letters_per_syll,0.0
phon_per_syll,0.0
letters_per_phon,0.0
letters_x_syll,0.0


FeatureSet-B missing ratio


,missing_ratio
Word,0.0
log_freq_pm,0.0
AoA_final,0.0
AoA_source,0.0
pos_group,0.0
Dom_PoS_SUBTLEX,0.0
lemma_length_diff,0.0
is_lemma_match,0.0
phon_x_syll,0.0
letters_x_syll,0.0


Age group distribution


,percentage
Age_Group,
age_8_plus,42.46
below_6,30.36
age_6_8,27.18


## 7. Export training-ready files

We export both CSV and Excel versions so the next modeling notebook can load them directly.



In [7]:
# Export the final training-ready files in both CSV and Excel formats.
feature_set_a_csv = OUTPUT_DIR / 'AoA_featureset_a_v3_freq2.csv'
feature_set_b_csv = OUTPUT_DIR / 'AoA_featureset_b_v3_freq2.csv'
feature_set_a_xlsx = OUTPUT_DIR / 'AoA_featureset_a_v3_freq2.xlsx'
feature_set_b_xlsx = OUTPUT_DIR / 'AoA_featureset_b_v3_freq2.xlsx'

feature_set_a_df.to_csv(feature_set_a_csv, index=False)
feature_set_b_df.to_csv(feature_set_b_csv, index=False)
feature_set_a_df.to_excel(feature_set_a_xlsx, index=False)
feature_set_b_df.to_excel(feature_set_b_xlsx, index=False)

print('Exported:')
print(feature_set_a_csv)
print(feature_set_b_csv)
print(feature_set_a_xlsx)
print(feature_set_b_xlsx)


Exported:
/Users/datong/Documents/5120/Nurodiversity inclusive design/data/TP10_DS/data/processed/iteration2/AoA_featureset_a_v3_freq2.csv
/Users/datong/Documents/5120/Nurodiversity inclusive design/data/TP10_DS/data/processed/iteration2/AoA_featureset_b_v3_freq2.csv
/Users/datong/Documents/5120/Nurodiversity inclusive design/data/TP10_DS/data/processed/iteration2/AoA_featureset_a_v3_freq2.xlsx
/Users/datong/Documents/5120/Nurodiversity inclusive design/data/TP10_DS/data/processed/iteration2/AoA_featureset_b_v3_freq2.xlsx


## 8. Notes for modeling

- Start with `FeatureSet-A` as the default baseline.
- Try `FeatureSet-B` when you want to test whether cleaned categorical signals improve the model.
- Do not use `AoA_Kup`, `AoA_Kup_lem`, `AoA_Bird_lem`, `AoA_Bristol_lem`, `AoA_Cort_lem`, or `AoA_Schock` as predictors if the target is `AoA_final` or `Age_Group`.

